# 🎮 Game Sales Analysis & Prediction — AIML Internship Project

**Dataset:** Video Game Sales Dataset (19,600 games)
**Goal:** End-to-end ML pipeline — Data Cleaning → EDA → Feature Engineering → Model Building → Evaluation

**Columns:** Rank, Name, Platform, Publisher, Developer, Critic_Score, User_Score, Total_Shipped (in millions), Year

---
### Project Steps
1. Import Libraries
2. Load Dataset
3. Data Understanding
4. Data Cleaning
5. Exploratory Data Analysis (EDA)
6. Feature Engineering
7. Train-Test Split
8. Model Building (Regression)
9. Model Evaluation
10. Feature Importance & Conclusion


## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")


## Step 2: Load Dataset

Note: File CSV format mein hai but UTF-8 encoding nahi use karti, isliye `encoding='latin1'` use kar rahe hain.

In [ ]:
df = pd.read_csv(r"C:\Users\karan\OneDrive\Desktop\game sales programming\game_sales_dataset.csv", encoding="latin1")
print("Shape of dataset:", df.shape)
df.head()


## Step 3: Data Understanding

Is step mein hum dataset ki basic information dekhenge — data types, statistical summary, aur missing values.

In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


In [ ]:
# Missing values check
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_percent.round(2)})
missing_df[missing_df["Missing Count"] > 0].sort_values("Missing %", ascending=False)


**Observation:** `Critic_Score` aur `User_Score` mein bahut zyada missing values hain (~50-90%). `Developer` mein sirf 2 missing values hain. `Total_Shipped` (jo humara target variable hoga) mein koi missing value nahi hai — yeh acchi baat hai.

## Step 4: Data Cleaning

1. Duplicate rows check aur remove
2. Missing `Developer` values fill
3. Missing `Critic_Score` / `User_Score` ko median se impute karenge
4. Data types verify karenge


In [ ]:
# 1. Check and remove duplicates
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()

# 2. Fill missing Developer with 'Unknown'
df["Developer"] = df["Developer"].fillna("Unknown")

# 3. Impute missing scores with median
df["Critic_Score"] = df["Critic_Score"].fillna(df["Critic_Score"].median())
df["User_Score"] = pd.to_numeric(df["User_Score"], errors="coerce")
df["User_Score"] = df["User_Score"].fillna(df["User_Score"].median())

# 4. Sanity check
print(df.isnull().sum().sum(), "missing values remaining")
df.head()


## Step 5: Exploratory Data Analysis (EDA)

### 5.1 Top 10 Publishers by Total Games Shipped

In [ ]:
top_publishers = df.groupby("Publisher")["Total_Shipped"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,6))
sns.barplot(x=top_publishers.values, y=top_publishers.index, palette="viridis")
plt.xlabel("Total Units Shipped (Millions)")
plt.title("Top 10 Publishers by Total Games Shipped")
plt.tight_layout()
plt.show()


### 5.2 Top 10 Platforms by Number of Games

In [ ]:
top_platforms = df["Platform"].value_counts().head(10)

plt.figure(figsize=(10,6))
sns.barplot(x=top_platforms.values, y=top_platforms.index, palette="mako")
plt.xlabel("Number of Games")
plt.title("Top 10 Platforms by Game Count")
plt.tight_layout()
plt.show()


### 5.3 Games Released Per Year (Trend)

In [ ]:
games_per_year = df[(df["Year"] >= 1980) & (df["Year"] <= 2020)]["Year"].value_counts().sort_index()

plt.figure(figsize=(12,6))
plt.plot(games_per_year.index, games_per_year.values, marker="o", color="darkorange")
plt.xlabel("Year")
plt.ylabel("Number of Games Released")
plt.title("Game Releases Over the Years")
plt.tight_layout()
plt.show()


### 5.4 Critic Score vs Total Shipped (Relationship)

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x="Critic_Score", y="Total_Shipped", alpha=0.4, color="teal")
plt.title("Critic Score vs Total Units Shipped")
plt.xlabel("Critic Score")
plt.ylabel("Total Shipped (Millions)")
plt.tight_layout()
plt.show()


### 5.5 Correlation Heatmap

In [ ]:
numeric_cols = ["Critic_Score", "User_Score", "Total_Shipped", "Year"]
plt.figure(figsize=(7,5))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


## Step 6: Feature Engineering

Model ke liye categorical columns (`Platform`, `Publisher`) ko numeric mein convert karenge using Label Encoding. `Name` aur `Developer` ko drop kar denge kyunki unke unique values bahut zyada hain (high cardinality) aur prediction mein direct useful nahi.

In [ ]:
model_df = df.copy()

le_platform = LabelEncoder()
le_publisher = LabelEncoder()

model_df["Platform_enc"] = le_platform.fit_transform(model_df["Platform"])
model_df["Publisher_enc"] = le_publisher.fit_transform(model_df["Publisher"])

features = ["Platform_enc", "Publisher_enc", "Critic_Score", "User_Score", "Year"]
target = "Total_Shipped"

X = model_df[features]
y = model_df[target]

X.head()


## Step 7: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)


## Step 8: Model Building (Regression)

Do models train karenge aur compare karenge:
1. **Linear Regression** (baseline model)
2. **Random Forest Regressor** (better for non-linear patterns)


In [ ]:
# Model 1: Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

# Model 2: Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print("Both models trained successfully!")


## Step 9: Model Evaluation

In [ ]:
def evaluate(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"--- {model_name} ---")
    print(f"MAE  : {mae:.3f}")
    print(f"RMSE : {rmse:.3f}")
    print(f"R2   : {r2:.3f}\n")
    return mae, rmse, r2

lr_metrics = evaluate(y_test, lr_preds, "Linear Regression")
rf_metrics = evaluate(y_test, rf_preds, "Random Forest Regressor")


In [ ]:
# Compare models visually
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [lr_metrics[0], rf_metrics[0]],
    "RMSE": [lr_metrics[1], rf_metrics[1]],
    "R2 Score": [lr_metrics[2], rf_metrics[2]]
})
results


In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=results, x="Model", y="R2 Score", palette="crest")
plt.title("Model Comparison: R2 Score")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


## Step 10: Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x=importances.values, y=importances.index, palette="flare")
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

importances


## Conclusion

- Dataset mein 19,600 games hain jinme sabse zyada games ka data Nintendo jaisi companies ka hai.
- `Critic_Score` aur `User_Score` mein bahut zyada missing values the jinhe median imputation se handle kiya gaya.
- Random Forest model ne Linear Regression ke comparison mein better R2 score diya, kyunki sales data mein non-linear patterns hote hain.
- Feature importance se pata chalta hai ki konsa feature (Platform, Publisher, Critic Score, etc.) sales predict karne mein sabse zyada contribute karta hai.

### Future Improvements
- More features add kiye ja sakte hain (genre, region-wise sales, agar available ho)
- Hyperparameter tuning (GridSearchCV) se model aur better ban sakta hai
- Advanced models try kiye ja sakte hain: XGBoost, Gradient Boosting
